#### 2026.07.09

In [4]:
# key
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [5]:
# OpenAI 연결
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

In [8]:
response = client.chat.completions.create(
    model='gpt-5.5',
    messages = [
        {
            'role' : 'user',
            'content' : '미니언즈 시리즈에 대해 설명해줘'
        }
    ]
)

In [12]:
response.choices[0].message.content

'**미니언즈 시리즈**는 일루미네이션 엔터테인먼트가 제작하고 유니버설 픽처스가 배급하는 애니메이션 프랜차이즈로, 노란색의 작은 생명체 **미니언들**과 악당에서 가족적인 인물로 변해가는 **그루**를 중심으로 전개되는 코미디 애니메이션 시리즈입니다.\n\n## 1. 미니언즈란?\n\n미니언들은 작고 노란 몸, 커다란 고글, 멜빵바지가 특징인 캐릭터들입니다.  \n이들은 오래전부터 존재해 왔으며, 본능적으로 “가장 강하고 나쁜 악당”을 섬기려는 습성을 가지고 있습니다.\n\n대표적인 특징은 다음과 같습니다.\n\n- 엉뚱하고 사고를 많이 침\n- 바나나를 매우 좋아함\n- 서로 닮았지만 개성이 있음\n- 이상한 언어인 **미니언어**, 즉 “미니언즈어”를 사용함\n- 순수하면서도 장난기 많고 코믹한 행동을 자주 함\n\n미니언즈어는 영어, 스페인어, 프랑스어, 이탈리아어, 일본어 등 여러 언어를 섞은 듯한 말투로, 정확한 의미보다는 소리와 상황으로 웃음을 주는 방식입니다.\n\n## 2. 시리즈의 시작: 《슈퍼배드》\n\n미니언들은 처음에는 영화 **《슈퍼배드》, 원제 Despicable Me**에서 주인공 그루의 부하로 등장했습니다.\n\n### 《슈퍼배드》, 2010\n\n주인공 **그루**는 세계 최고의 악당이 되려는 인물입니다.  \n그는 달을 훔치겠다는 계획을 세우지만, 세 명의 고아 소녀 **마고, 에디트, 아그네스**를 만나면서 점점 따뜻한 마음을 가진 아버지로 변하게 됩니다.\n\n이 영화에서 미니언들은 그루의 실험실에서 일하는 조수이자 부하들로 등장했고, 특유의 귀여움과 코미디로 큰 인기를 얻었습니다.\n\n## 3. 주요 영화 목록\n\n개봉 순서로 보면 다음과 같습니다.\n\n1. **《슈퍼배드》, Despicable Me, 2010**\n2. **《슈퍼배드 2》, Despicable Me 2, 2013**\n3. **《미니언즈》, Minions, 2015**\n4. **《슈퍼배드 3》, Despicable Me 3, 2017**\n5. *

Function Calling

In [3]:
# pdf 26p
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

# pdf 27p
tools = [
    {
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
        },
        "strict": True
    }
]

In [4]:
import json

input_message = [
    {
        'role':'user',
        'content':'오늘 파리의 날씨는 어때?'
    }
]

response = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)


In [5]:
tool_call = response.output[1]
print(tool_call)
args = json.loads(tool_call.arguments)
result = get_weather(**args)
print("2. 로컬 함수 실행 결과:", result)

ResponseFunctionToolCall(arguments='{"latitude":48.8566,"longitude":2.3522}', call_id='call_kBYIRk8uFkD06nTil9mntvga', name='get_weather', type='function_call', id='fc_0a74dc7e64ad720b006a4f287eeea8819b84fc694fbb0c0b6a', namespace=None, status='completed')
2. 로컬 함수 실행 결과: 22.5


In [6]:
input_message.append(tool_call)

input_message.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

response_2 = client.responses.create(
    model="gpt-4.1",
    input=input_message,
    tools=tools
)

print("3. 모델의 최종 자연어 답변:\n", response_2.output_text)

3. 모델의 최종 자연어 답변:
 오늘 파리의 기온은 약 22.5도입니다. 날씨는 쾌적할 것으로 보입니다! 추가적으로 궁금한 점이나 날씨 상세 정보가 필요하다면 말씀해 주세요.


웹 검색

In [4]:
response = client.responses.create(
    model='gpt-5.5',
    tools=[
        {
            'type' : 'web_search'
        }
    ],
    input='최근 환율에 대한 이슈가 있어?'
)

In [5]:
print(response.output_text)

네. **원/달러 환율 기준으로는 최근 이슈가 꽤 큽니다.** 핵심은 “급등 후 급락”, “24시간 외환시장 개장”, “당국 개입/모니터링”, “외국인 자금 흐름”입니다.

1. **원/달러 환율 변동성이 커졌습니다.**  
   7월 2일에는 원/달러 환율이 **1,555.8원** 수준까지 올라가며 원화 약세가 이어졌고, 이는 외국인의 국내 주식 순매도와 연결돼 설명됐습니다. 당시 원화는 2009년 3월 이후 가장 약한 수준에 근접했다는 보도도 있었습니다. ([en.yna.co.kr](https://en.yna.co.kr/view/AEN20260702003151320))

2. **그런데 7월 8일에는 1,500원 아래로 급락했습니다.**  
   7월 8일 오후 3시 30분 기준 원/달러 환율은 **1,498원**으로 내려와, 5월 29일 이후 처음으로 1,400원대에 진입했습니다. SK하이닉스의 미국 ADR 상장·주식 발행 관련 달러 유입 기대가 원화 강세 요인으로 언급됐습니다. ([m.ajupress.com](https://m.ajupress.com/view/20260708165902538))

3. **한국 외환시장이 7월 6일부터 24시간 거래 체제로 바뀐 것도 큰 이슈입니다.**  
   정부는 2026년 7월 6일 24시간 외환시장 개장을 공식 발표했고, 이를 원화의 글로벌 접근성을 높이는 외환시장 개혁으로 설명했습니다. 다만 정부와 한국은행은 새 제도 도입 이후 시장 영향을 면밀히 모니터링하겠다고 밝혔습니다. ([english.mofe.go.kr](https://english.mofe.go.kr/pc/selectTbPressCenterDtl.do?boardCd=N0001&seq=6439))

4. **외환당국의 시장 안정 개입도 주목됩니다.**  
   한국 외환당국은 2026년 1분기에 외환시장 안정을 위해 **순 136억 달러**를 매도한 것으로 집계됐고, 2024년 4분기 이후 6개 분기 연속 순매도였습니다. 이는 원화 약세를 막기 위한 

File Search

In [2]:
import requests
from io import BytesIO

def create_file(client, file_path):
    if file_path.startswith("http://") or file_path.startswith("https://"):
        # Download the file content from the URL
        response = requests.get(file_path)
        file_content = BytesIO(response.content)
        file_name = file_path.split("/")[-1]
        file_tuple = (file_name, file_content)
        result = client.files.create(
            file=file_tuple,
            purpose="assistants"
        )
    else:
        # Handle local file path
        with open(file_path, "rb") as file_content:
            result = client.files.create(
                file=file_content,
                purpose="assistants"
            )
    print(result.id)
    return result.id

In [6]:
# file 객체 생성 : create_file
file_id = create_file(client, 'howto-sockets.pdf')

file-S7ctQqaeWiys63KjBtntJA


In [7]:
# 벡터스토어 생성
vector_store = client.vector_stores.create(
    name='know_base'
)

In [8]:
vector_store

VectorStore(id='vs_6a5057f44b208191b284f315d64d271f', created_at=1783650292, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1783650292, metadata={}, name='know_base', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None)

In [9]:
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_id
)

VectorStoreFile(id='file-S7ctQqaeWiys63KjBtntJA', created_at=1783650297, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_6a5057f44b208191b284f315d64d271f', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [10]:
file_list = client.vector_stores.files.list(
    vector_store_id=vector_store.id
)

In [11]:
file_list

SyncCursorPage[VectorStoreFile](data=[], has_more=False, object='list', first_id=None, last_id=None)

In [12]:
input = '파이썬 코드로 소켓을 만드는 방법을 간단히 설명해줘'

response = client.responses.create(
    model='gpt-5.5',
    input=input,
    tools=[
        {
            'type':'file_search',
            'vector_store_ids': [vector_store.id]
        }
    ]
)

In [13]:
print(response.output_text)

파이썬에서는 내장 모듈인 `socket`을 사용해서 소켓을 만들 수 있습니다.

## 1. TCP 서버 소켓 예시

```python
import socket

# 소켓 생성: IPv4, TCP
server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 주소와 포트 바인딩
server_socket.bind(("127.0.0.1", 5000))

# 클라이언트 접속 대기
server_socket.listen()

print("서버 대기 중...")

# 클라이언트 연결 수락
client_socket, addr = server_socket.accept()
print("연결됨:", addr)

# 데이터 받기
data = client_socket.recv(1024)
print("받은 데이터:", data.decode())

# 데이터 보내기
client_socket.send("Hello Client".encode())

# 소켓 닫기
client_socket.close()
server_socket.close()
```

## 2. TCP 클라이언트 소켓 예시

```python
import socket

# 소켓 생성
client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 서버에 연결
client_socket.connect(("127.0.0.1", 5000))

# 데이터 보내기
client_socket.send("Hello Server".encode())

# 데이터 받기
data = client_socket.recv(1024)
print("서버 응답:", data.decode())

# 소켓 닫기
client_socket.close()
```

## 핵심 흐름

### 서버
```text
socket() → bind() → listen() → accept() → recv()/send() → close()
```

### 클라이언트
